# PREDICT-Play | XGBoost

Run this notebook after `02_feature_engineering.ipynb`.

## Imports and load data

In [1]:
# Import the modelling libraries.
from pathlib import Path
import copy
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

np.random.seed(42)


In [2]:
# Load the shared title features and graph files created in notebook 02.
model_features = pd.read_csv( "https://raw.githubusercontent.com/bjvvuuren/predict-play/refs/heads/main/modelling/datasets/feature-engineering/model_features.csv")

In [3]:
# Identify the feature columns and row indices used by both models.
identifier_columns = {
    "node_id",
    "movieId",
    "title_id",
    "canonical_title",
    "role",
    "target"
}

feature_columns = [
    column
    for column in model_features.columns
    if column not in identifier_columns
]

X = model_features[feature_columns].to_numpy(dtype=np.float32)
y = model_features["target"].to_numpy(dtype=np.int64)
roles = model_features["role"].to_numpy()

train_index = np.flatnonzero(roles == "train")
tune_index = np.flatnonzero(roles == "tune")
calibration_index = np.flatnonzero(roles == "calibration")
catalogue_index = np.flatnonzero(roles == "catalogue")

positive_weight = float((y[train_index] == 0).sum()/(y[train_index] == 1).sum())


## XGBoost

XGBoost receives the same engineered title feature columns used on the movie nodes. It does not receive graph edges. The tuning split controls early stopping, while the calibration split remains untouched until probability calibration.

In [4]:
# Configure and train XGBoost.
xgb = XGBClassifier(
    objective="binary:logistic",
    tree_method="hist",
    n_estimators=800,
    max_depth=4,
    learning_rate=0.04,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=5.0,
    scale_pos_weight=positive_weight,
    eval_metric="logloss",
    early_stopping_rounds=40,
    n_jobs=4,
    random_state=42
)

xgb_training_start = time.perf_counter()

xgb.fit(
    X[train_index],
    y[train_index],
    eval_set=[
        (X[tune_index], y[tune_index])
    ],
    verbose=False
)

xgb_training_seconds = time.perf_counter() - xgb_training_start

xgb_tune_probability = xgb.predict_proba(
    X[tune_index]
)[:, 1]

xgb_tune_ap = average_precision_score(
    y[tune_index],
    xgb_tune_probability
)

xgb.save_model("xgboost.ubj")

print("Best XGBoost iteration:", xgb.best_iteration)
print("Tuning AP:", round(xgb_tune_ap, 4))
print("Training seconds:", round(xgb_training_seconds, 2))


Best XGBoost iteration: 343
Tuning AP: 0.4741
Training seconds: 6.29


In [5]:
# Produce raw XGBoost margins for every title using the selected boosting iteration.
xgb_margins = xgb.predict(
    X,
    output_margin=True,
    iteration_range=(0, xgb.best_iteration + 1)
)


## Calibrate probability outputs

This is probability post-processing, not a third competing predictive model.

In [6]:
# Fit one sigmoid calibrator per model using only the calibration subset.
xgb_calibrator = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=1000
)

xgb_calibrator.fit(
    xgb_margins[calibration_index].reshape(-1, 1),
    y[calibration_index]
)

xgb_probabilities = xgb_calibrator.predict_proba(
    xgb_margins.reshape(-1, 1)
)[:, 1]

joblib.dump(
    xgb_calibrator,"xgboost_calibrator.joblib"
)


['xgboost_calibrator.joblib']

## Compute tuning performance and score the acquisition catalogue

In [7]:
# Compare the two models on the same untouched tuning titles.

xgb_tune_probability = xgb_probabilities[tune_index]

model_summary = pd.DataFrame({
    "model": "XGBoost",
    "tuning_average_precision":
        average_precision_score(
            y[tune_index],
            xgb_tune_probability
        ),
    "training_seconds": [
        xgb_training_seconds
    ]
})

model_summary.to_csv("xgboost_model_summary.csv",
    index=False
)

display(model_summary)


,model,tuning_average_precision,training_seconds
0,XGBoost,0.47415,6.28696


In [8]:
tuning_prevalence = y[tune_index].mean()

print("Tuning positive rate:", round(tuning_prevalence, 4))
print("XGBoost AP:", round(xgb_tune_ap, 4))
print(
    "XGBoost lift over baseline:",
    round(xgb_tune_ap / tuning_prevalence, 2)
)

Tuning positive rate: 0.2353
XGBoost AP: 0.4741
XGBoost lift over baseline: 2.01


In [9]:
# Export acquisition-catalogue probabilities without attaching any catalogue outcome.
predictions = model_features.loc[
    catalogue_index,
    [
        "movieId",
        "title_id",
        "canonical_title"
    ]
].copy()

predictions["xgboost_p_meets_engagement_threshold"] = (
    xgb_probabilities[catalogue_index]
)

predictions["prediction_basis"] = (
    "synthetic_tmdb_popularity_benchmarked_engagement"
)

predictions.to_csv("xgboost_catalogue_predictions.csv",
    index=False
)

display(
    predictions.sort_values(
        "xgboost_p_meets_engagement_threshold",
        ascending=False
    ).head(10)
)


,movieId,title_id,canonical_title,xgboost_p_meets_engagement_threshold,prediction_basis
6260,6365,STAD-MOV-006365,"Matrix Reloaded, The",0.912207,synthetic_tmdb_popularity_benchmarked_engagement
6822,6934,STAD-MOV-006934,"Matrix Revolutions, The",0.912207,synthetic_tmdb_popularity_benchmarked_engagement
11826,52722,STAD-MOV-052722,Spider-Man 3,0.896728,synthetic_tmdb_popularity_benchmarked_engagement
9582,30793,STAD-MOV-030793,Charlie and the Chocolate Factory,0.895972,synthetic_tmdb_popularity_benchmarked_engagement
11357,48414,STAD-MOV-048414,Open Season,0.893797,synthetic_tmdb_popularity_benchmarked_engagement
11452,49274,STAD-MOV-049274,Happy Feet,0.893797,synthetic_tmdb_popularity_benchmarked_engagement
12460,58105,STAD-MOV-058105,"Spiderwick Chronicles, The",0.892660,synthetic_tmdb_popularity_benchmarked_engagement
20615,101076,STAD-MOV-101076,G.I. Joe: Retaliation,0.888727,synthetic_tmdb_popularity_benchmarked_engagement
255,258,STAD-MOV-000258,"Kid in King Arthur's Court, A",0.887390,synthetic_tmdb_popularity_benchmarked_engagement
2540,2625,STAD-MOV-002625,Black Mask (Hak hap),0.886403,synthetic_tmdb_popularity_benchmarked_engagement


In [10]:
# Export tuning predictions for later Part C metric calculations.
tuning_predictions = model_features.loc[
    tune_index,
    [
        "movieId",
        "title_id",
        "canonical_title",
        "target"
    ]
].copy()


tuning_predictions["xgboost_probability"] = (
    xgb_probabilities[tune_index]
)

tuning_predictions.to_csv("xgboost_tuning_predictions.csv",
    index=False
)


## Leakage boundary

Neither model receives `tmdb_popularity`, MovieLens rating values, platform household counts, qualifying household counts, watch durations or synthetic search demand as predictors. TMDB popularity was used in preprocessing to generate the synthetic viewing outcome, so allowing it into either model would directly leak the target-generating mechanism.

The acquisition catalogue is scored from metadata only. Its targets are not present in this notebook.